# 2D Transformations and Image Warping

This notebook is a self-study chapter for computer vision learners. It follows a **concept first, code second** structure:

1. Read the concept and formulas.
2. Run the code cell immediately after the concept.
3. Complete the small task or self-check.
4. Modify parameters and explain what changes.

By the end, you should be able to:

- Explain the difference between filtering and warping.
- Represent scale, shear, rotation, translation, affine transforms, and homographies with matrices.
- Use homogeneous coordinates to combine transformations.
- Estimate an affine transform from point correspondences using least squares.
- Explain forward warping, inverse warping, and bilinear interpolation.
- Build a small image rectification pipeline from four point correspondences.

Recommended workflow: run every code cell from top to bottom once, then return to each **Task** and experiment.

## 0. Setup

We will use only common Python libraries. The synthetic images are generated inside this notebook, so no external image files are required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

# Keep plots compact and readable in notebooks.
plt.rcParams["figure.figsize"] = (6, 5)
plt.rcParams["axes.grid"] = True

## 1. Images as Functions: Filtering vs. Warping

A grayscale image can be viewed as a 2D function:

$$
f(x, y): \Omega \subset \mathbb{R}^2 \rightarrow \mathbb{R}
$$

Here, $(x, y)$ is a pixel location and $f(x, y)$ is the intensity value at that location.

Computer vision often uses two major kinds of image transformations:

- **Filtering** changes the range of the image function. Pixel values change, but pixel locations stay fixed.
- **Warping** changes the domain of the image function. Pixel locations move, so the image geometry changes.

For a geometric transform $T$, warping maps coordinates between images:

$$
(x', y') = T(x, y)
$$

The key idea in this chapter is that many useful geometric transformations can be written as matrices.

In [ ]:
def make_synthetic_image(height=160, width=220):
    # Create a synthetic grayscale image with edges, gradients, and simple shapes.
    yy, xx = np.mgrid[0:height, 0:width]
    img = 0.15 + 0.45 * (xx / max(width - 1, 1)) + 0.25 * (yy / max(height - 1, 1))

    # Add a rectangle.
    img[35:115, 55:150] += 0.35

    # Add a circular disk.
    cy, cx, r = 95, 165, 28
    mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2
    img[mask] = 0.05

    # Add diagonal stripes to make warping easier to see.
    stripes = ((xx + yy) // 16) % 2 == 0
    img[stripes] += 0.12

    return np.clip(img, 0, 1)

image = make_synthetic_image()

plt.imshow(image, cmap="gray", origin="upper")
plt.title("Synthetic grayscale image")
plt.xlabel("x coordinate")
plt.ylabel("y coordinate")
plt.colorbar(label="intensity")
plt.show()

### Task 1: Coordinate intuition

1. Change `height` and `width` in `make_synthetic_image`.
2. Observe how the x-axis and y-axis labels change.
3. Explain why image coordinates usually place the origin in the upper-left corner, while mathematical plots often place the origin in the lower-left corner.

## 2. Linear 2D Transformations

A 2D linear transformation maps a point $p = (x, y)^T$ to a new point $p' = (x', y')^T$ by matrix multiplication:

$$
p' = A p
$$

where

$$
A =
\begin{bmatrix}
a & b \\
c & d
\end{bmatrix}
$$

Important examples include scaling, shearing, and rotation.

### Scaling

$$
\begin{bmatrix}
x' \\
y'
\end{bmatrix}
=
\begin{bmatrix}
s_x & 0 \\
0 & s_y
\end{bmatrix}
\begin{bmatrix}
x \\
y
\end{bmatrix}
$$

If $s_x = s_y$, the scaling is **uniform**. If $s_x \neq s_y$, the object changes aspect ratio.

### Shearing

A horizontal shear has the form:

$$
\begin{bmatrix}
x' \\
y'
\end{bmatrix}
=
\begin{bmatrix}
1 & k_x \\
0 & 1
\end{bmatrix}
\begin{bmatrix}
x \\
y
\end{bmatrix}
$$

A vertical shear has the form:

$$
\begin{bmatrix}
x' \\
y'
\end{bmatrix}
=
\begin{bmatrix}
1 & 0 \\
k_y & 1
\end{bmatrix}
\begin{bmatrix}
x \\
y
\end{bmatrix}
$$

### Rotation around the origin

Using the angle $\theta$ in radians:

$$
R(\theta) =
\begin{bmatrix}
\cos\theta & -\sin\theta \\
\sin\theta & \cos\theta
\end{bmatrix}
$$

Then

$$
p' = R(\theta)p
$$

In [ ]:
def make_grid_points(nx=7, ny=7, spacing=1.0):
    # Create a centered 2D point grid.
    x = (np.arange(nx) - (nx - 1) / 2) * spacing
    y = (np.arange(ny) - (ny - 1) / 2) * spacing
    xx, yy = np.meshgrid(x, y)
    return np.column_stack([xx.ravel(), yy.ravel()])


def apply_linear(A, points):
    # Apply a 2x2 linear transform to Nx2 points.
    return points @ A.T


def plot_points(before, after, title="2D point transformation"):
    # Plot points before and after a transformation.
    plt.figure(figsize=(6, 6))
    plt.scatter(before[:, 0], before[:, 1], label="original")
    plt.scatter(after[:, 0], after[:, 1], label="transformed")

    # Draw arrows from original points to transformed points.
    delta = after - before
    plt.quiver(before[:, 0], before[:, 1], delta[:, 0], delta[:, 1],
               angles="xy", scale_units="xy", scale=1, width=0.004)

    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.axis("equal")
    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.show()

points = make_grid_points()

S = np.array([[1.6, 0.0],
              [0.0, 0.7]])

shear_x = np.array([[1.0, 0.5],
                    [0.0, 1.0]])

theta = np.deg2rad(35)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

plot_points(points, apply_linear(S, points), "Non-uniform scaling")
plot_points(points, apply_linear(shear_x, points), "Horizontal shear")
plot_points(points, apply_linear(R, points), "Rotation around the origin")

### Task 2: Linear transform experiments

1. Set `S = np.array([[2.0, 0.0], [0.0, 2.0]])`. What is preserved?
2. Set `S = np.array([[2.0, 0.0], [0.0, 0.5]])`. What changes?
3. Try a negative scale factor, such as `S = np.array([[-1.0, 0.0], [0.0, 1.0]])`. Which axis is flipped?
4. Change the rotation angle to `90`, `180`, and `-45` degrees. Explain the sign convention.

## 3. Homogeneous Coordinates and Translation

A pure 2D translation is not a linear transformation in standard 2D coordinates, because it adds a constant offset:

$$
\begin{bmatrix}
x' \\
y'
\end{bmatrix}
=
\begin{bmatrix}
x \\
y
\end{bmatrix}
+
\begin{bmatrix}
t_x \\
t_y
\end{bmatrix}
$$

To represent translation as matrix multiplication, we use **homogeneous coordinates**. A 2D point $(x, y)$ becomes a 3D vector:

$$
\tilde{p} =
\begin{bmatrix}
x \\
y \\
1
\end{bmatrix}
$$

The translation matrix is:

$$
T(t_x, t_y) =
\begin{bmatrix}
1 & 0 & t_x \\
0 & 1 & t_y \\
0 & 0 & 1
\end{bmatrix}
$$

Then

$$
\tilde{p}' = T(t_x, t_y)\tilde{p}
$$

Homogeneous vectors are defined up to scale:

$$
\begin{bmatrix}
x \\
y \\
1
\end{bmatrix}
\sim
\begin{bmatrix}
wx \\
wy \\
w
\end{bmatrix}
$$

To convert from homogeneous to standard coordinates, divide by the last coordinate:

$$
(x, y) = \left(\frac{X}{W}, \frac{Y}{W}\right)
$$

In [ ]:
def to_homogeneous(points):
    # Convert Nx2 points to Nx3 homogeneous points.
    ones = np.ones((points.shape[0], 1))
    return np.hstack([points, ones])


def from_homogeneous(points_h):
    # Convert Nx3 homogeneous points to Nx2 points.
    w = points_h[:, 2:3]
    if np.any(np.isclose(w, 0)):
        raise ValueError("Cannot convert points with w close to zero.")
    return points_h[:, :2] / w


def apply_homography(H, points):
    # Apply a 3x3 homogeneous transformation to Nx2 points.
    points_h = to_homogeneous(points)
    transformed_h = points_h @ H.T
    return from_homogeneous(transformed_h)


def translation(tx, ty):
    # Create a 3x3 translation matrix.
    return np.array([[1.0, 0.0, tx],
                     [0.0, 1.0, ty],
                     [0.0, 0.0, 1.0]])


def scaling(sx, sy):
    # Create a 3x3 scaling matrix.
    return np.array([[sx, 0.0, 0.0],
                     [0.0, sy, 0.0],
                     [0.0, 0.0, 1.0]])


def rotation(theta_rad):
    # Create a 3x3 rotation matrix around the origin.
    c, s = np.cos(theta_rad), np.sin(theta_rad)
    return np.array([[c, -s, 0.0],
                     [s,  c, 0.0],
                     [0.0, 0.0, 1.0]])


def shear(kx=0.0, ky=0.0):
    # Create a 3x3 shear matrix.
    return np.array([[1.0, kx, 0.0],
                     [ky, 1.0, 0.0],
                     [0.0, 0.0, 1.0]])

H_translate = translation(2.0, -1.0)
translated = apply_homography(H_translate, points)
plot_points(points, translated, "Translation using homogeneous coordinates")

### Task 3: Homogeneous coordinate self-check

Run the following checks mentally first, then verify them in code:

1. What point does $(2, 3)$ become after `translation(10, -4)`?
2. What does `from_homogeneous(np.array([[4, 6, 2]]))` return?
3. Why is `np.array([[4, 6, 2]])` the same 2D point as `np.array([[2, 3, 1]])` in homogeneous coordinates?

In [ ]:
# Self-check examples.
p = np.array([[2.0, 3.0]])
print("Translated point:", apply_homography(translation(10, -4), p))
print("Homogeneous conversion:", from_homogeneous(np.array([[4.0, 6.0, 2.0]])))

## 4. Matrix Composition: Does Order Matter?

Homogeneous transformations can be combined by matrix multiplication.

If we first apply $H_1$ and then apply $H_2$, the combined transform is:

$$
\tilde{p}' = H_2 H_1 \tilde{p}
$$

Matrix multiplication is generally **not commutative**:

$$
H_2 H_1 \neq H_1 H_2
$$

This means that rotating and then translating is usually different from translating and then rotating.

In [ ]:
H_rotate = rotation(np.deg2rad(45))
H_shift = translation(2.5, 0.5)

# First rotate, then translate.
H_rotate_then_shift = H_shift @ H_rotate

# First translate, then rotate.
H_shift_then_rotate = H_rotate @ H_shift

out1 = apply_homography(H_rotate_then_shift, points)
out2 = apply_homography(H_shift_then_rotate, points)

plot_points(points, out1, "First rotate, then translate")
plot_points(points, out2, "First translate, then rotate")

print("H_shift @ H_rotate =\n", H_rotate_then_shift)
print("H_rotate @ H_shift =\n", H_shift_then_rotate)

### Task 4: Composition challenge

Create a transform that rotates points around the point $(2, 1)$ instead of around the origin.

Hint: use three steps.

$$
H = T(c_x, c_y) R(\theta) T(-c_x, -c_y)
$$

Then test your result on a point that is exactly at the center $(2, 1)$. It should not move.

In [ ]:
# Student exercise scaffold: rotate around an arbitrary center.
center = np.array([2.0, 1.0])
angle = np.deg2rad(60)

H_center_rotation = translation(center[0], center[1]) @ rotation(angle) @ translation(-center[0], -center[1])

center_as_point = center.reshape(1, 2)
print("Center before:", center_as_point)
print("Center after: ", apply_homography(H_center_rotation, center_as_point))

## 5. Classification of 2D Transformations

Different 2D transformation families have different constraints and degrees of freedom.

### Translation: 2 DOF

$$
H =
\begin{bmatrix}
1 & 0 & t_x \\
0 & 1 & t_y \\
0 & 0 & 1
\end{bmatrix}
$$

### Euclidean or rigid transform: 3 DOF

Rotation plus translation. Distances and angles are preserved.

$$
H =
\begin{bmatrix}
\cos\theta & -\sin\theta & t_x \\
\sin\theta & \cos\theta & t_y \\
0 & 0 & 1
\end{bmatrix}
$$

### Similarity transform: 4 DOF

Uniform scale, rotation, and translation. Angles are preserved, but distances scale by $s$.

$$
H =
\begin{bmatrix}
s\cos\theta & -s\sin\theta & t_x \\
s\sin\theta & s\cos\theta & t_y \\
0 & 0 & 1
\end{bmatrix}
$$

### Affine transform: 6 DOF

Arbitrary linear transform plus translation. Parallel lines remain parallel.

$$
H =
\begin{bmatrix}
a & b & t_x \\
c & d & t_y \\
0 & 0 & 1
\end{bmatrix}
$$

### Projective transform or homography: 8 DOF

A general $3 \times 3$ matrix defined up to scale. Lines remain lines, but parallel lines may meet.

$$
H =
\begin{bmatrix}
h_{11} & h_{12} & h_{13} \\
h_{21} & h_{22} & h_{23} \\
h_{31} & h_{32} & h_{33}
\end{bmatrix}
$$

Because homogeneous matrices are defined up to scale, the homography has $9 - 1 = 8$ degrees of freedom.

In [ ]:
def square_with_diagonals():
    # Create points that trace a square and its two diagonals.
    square = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1], [-1, -1]], dtype=float)
    diag1 = np.array([[-1, -1], [1, 1]], dtype=float)
    diag2 = np.array([[-1, 1], [1, -1]], dtype=float)
    return [square, diag1, diag2]


def plot_shape(lines, H=None, title="Transformed shape"):
    # Plot original and transformed line segments.
    plt.figure(figsize=(6, 6))
    for index, line in enumerate(lines):
        plt.plot(line[:, 0], line[:, 1], marker="o", label="original" if index == 0 else None)
        if H is not None:
            out = apply_homography(H, line)
            plt.plot(out[:, 0], out[:, 1], marker="o", linestyle="--", label="transformed" if index == 0 else None)
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.axis("equal")
    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.show()

lines = square_with_diagonals()

H_affine = translation(1.0, 0.3) @ shear(kx=0.7) @ scaling(1.2, 0.8) @ rotation(np.deg2rad(20))

H_projective = np.array([[1.0, 0.2, 0.0],
                         [0.1, 1.0, 0.0],
                         [0.25, 0.15, 1.0]])

plot_shape(lines, H_affine, "Affine transform: parallel lines stay parallel")
plot_shape(lines, H_projective, "Projective transform: perspective-like distortion")

### Task 5: Transformation classification

For each matrix below, classify it as translation, Euclidean, similarity, affine, or projective.

$$
H_1 =
\begin{bmatrix}
1 & 0 & 3 \\
0 & 1 & -2 \\
0 & 0 & 1
\end{bmatrix}
$$

$$
H_2 =
\begin{bmatrix}
0 & -1 & 4 \\
1 & 0 & 5 \\
0 & 0 & 1
\end{bmatrix}
$$

$$
H_3 =
\begin{bmatrix}
2 & 0 & 4 \\
0 & 2 & 5 \\
0 & 0 & 1
\end{bmatrix}
$$

$$
H_4 =
\begin{bmatrix}
1 & 0.4 & 4 \\
0.2 & 1 & 5 \\
0 & 0 & 1
\end{bmatrix}
$$

$$
H_5 =
\begin{bmatrix}
1 & 0.4 & 4 \\
0.2 & 1 & 5 \\
0.001 & 0.002 & 1
\end{bmatrix}
$$

## 6. Estimating an Affine Transform from Point Correspondences

Suppose we have matched points in two images:

$$
(x_i, y_i) \rightarrow (u_i, v_i)
$$

An affine transform has the form:

$$
u_i = a x_i + b y_i + t_x
$$

$$
v_i = c x_i + d y_i + t_y
$$

The unknown parameter vector is:

$$
\theta =
\begin{bmatrix}
a & b & t_x & c & d & t_y
\end{bmatrix}^T
$$

Each point correspondence gives two linear equations:

$$
\begin{bmatrix}
x_i & y_i & 1 & 0 & 0 & 0 \\
0 & 0 & 0 & x_i & y_i & 1
\end{bmatrix}
\theta
=
\begin{bmatrix}
u_i \\
v_i
\end{bmatrix}
$$

With at least 3 non-collinear point correspondences, we can solve for the 6 affine parameters. With more than 3 correspondences, we usually solve a least-squares problem:

$$
\hat{\theta} = \arg\min_{\theta} \lVert A\theta - b \rVert_2^2
$$

The normal-equation form is:

$$
\hat{\theta} = (A^T A)^{-1} A^T b
$$

In practice, use a numerical least-squares solver instead of explicitly computing the inverse.

In [ ]:
def estimate_affine(src, dst):
    # Estimate a 3x3 affine matrix from Nx2 point correspondences.
    if src.shape != dst.shape or src.shape[1] != 2:
        raise ValueError("src and dst must both have shape Nx2.")
    if src.shape[0] < 3:
        raise ValueError("At least 3 point correspondences are required.")

    A_rows = []
    b_rows = []
    for (x, y), (u, v) in zip(src, dst):
        A_rows.append([x, y, 1, 0, 0, 0])
        A_rows.append([0, 0, 0, x, y, 1])
        b_rows.append(u)
        b_rows.append(v)

    A = np.asarray(A_rows, dtype=float)
    b = np.asarray(b_rows, dtype=float)

    theta, residuals, rank, singular_values = np.linalg.lstsq(A, b, rcond=None)
    H = np.array([[theta[0], theta[1], theta[2]],
                  [theta[3], theta[4], theta[5]],
                  [0.0,      0.0,      1.0]])
    return H, residuals, rank, singular_values

# Create source points.
src = np.array([[-1.0, -1.0],
                [ 1.0, -1.0],
                [ 1.0,  1.0],
                [-1.0,  1.0],
                [ 0.0,  0.0],
                [ 0.5, -0.2]])

# Apply a known affine transform and add small measurement noise.
H_true = translation(2.0, -0.7) @ shear(kx=0.35) @ scaling(1.4, 0.8) @ rotation(np.deg2rad(25))
rng = np.random.default_rng(7)
dst_clean = apply_homography(H_true, src)
dst_noisy = dst_clean + rng.normal(scale=0.025, size=dst_clean.shape)

H_est, residuals, rank, singular_values = estimate_affine(src, dst_noisy)
dst_pred = apply_homography(H_est, src)

print("True affine matrix:\n", H_true)
print("Estimated affine matrix:\n", H_est)
print("Residuals from np.linalg.lstsq:", residuals)
print("Rank of system:", rank)

plot_points(dst_noisy, dst_pred, "Measured target points vs. affine predictions")

### Task 6: Least-squares understanding

1. Why does each point correspondence produce two equations?
2. Why do we need at least 3 non-collinear point correspondences for an affine transform?
3. Add one bad correspondence to `dst_noisy` and estimate the transform again. What happens?
4. Compute the residual vector $r = A\theta - b$ manually and report the mean squared residual.

In [ ]:
def affine_design_matrix(src, dst):
    # Build the linear system A theta = b for affine estimation.
    A_rows = []
    b_rows = []
    for (x, y), (u, v) in zip(src, dst):
        A_rows.append([x, y, 1, 0, 0, 0])
        A_rows.append([0, 0, 0, x, y, 1])
        b_rows.append(u)
        b_rows.append(v)
    return np.asarray(A_rows), np.asarray(b_rows)

A_affine, b_affine = affine_design_matrix(src, dst_noisy)
theta_est = np.array([H_est[0, 0], H_est[0, 1], H_est[0, 2],
                      H_est[1, 0], H_est[1, 1], H_est[1, 2]])
residual_vector = A_affine @ theta_est - b_affine
mse = np.mean(residual_vector ** 2)

print("Residual vector:", residual_vector)
print("Mean squared residual:", mse)

## 7. Projective Transformations and Homographies

A projective transformation, also called a **homography**, maps homogeneous points by:

$$
\lambda
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}
=
H
\begin{bmatrix}
x \\
y \\
1
\end{bmatrix}
$$

where $H$ is a $3 \times 3$ matrix and $\lambda$ is an arbitrary nonzero scale.

A homography can model perspective effects for planar surfaces, such as a document page, a poster, a road plane, or a wall.

Key properties:

- Lines map to lines.
- Parallel lines may no longer remain parallel.
- Ratios of distances are not generally preserved.
- The matrix has 8 degrees of freedom because it is defined up to scale.

To estimate a homography from point correspondences, at least 4 point correspondences are required.

In [ ]:
def estimate_homography(src, dst):
    # Estimate a homography using the Direct Linear Transform algorithm.
    if src.shape != dst.shape or src.shape[1] != 2:
        raise ValueError("src and dst must both have shape Nx2.")
    if src.shape[0] < 4:
        raise ValueError("At least 4 correspondences are required for a homography.")

    A = []
    for (x, y), (u, v) in zip(src, dst):
        A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    A = np.asarray(A, dtype=float)

    # The solution is the right singular vector associated with the smallest singular value.
    _, _, Vt = np.linalg.svd(A)
    h = Vt[-1]
    H = h.reshape(3, 3)

    # Normalize for a stable display.
    if not np.isclose(H[2, 2], 0):
        H = H / H[2, 2]
    return H

src_quad = np.array([[0.0, 0.0],
                     [3.0, 0.0],
                     [3.0, 2.0],
                     [0.0, 2.0]])

dst_quad = np.array([[0.2, 0.1],
                     [2.8, -0.2],
                     [3.4, 2.2],
                     [-0.3, 1.7]])

H_quad = estimate_homography(src_quad, dst_quad)
print("Estimated homography from four corners:\n", H_quad)
print("Mapped source corners:\n", apply_homography(H_quad, src_quad))
print("Target corners:\n", dst_quad)

# Visualize the mapping.
plt.figure(figsize=(6, 5))
plt.plot(*np.vstack([src_quad, src_quad[0]]).T, marker="o", label="source quadrilateral")
plt.plot(*np.vstack([dst_quad, dst_quad[0]]).T, marker="o", label="target quadrilateral")
plt.axis("equal")
plt.title("Four-point homography mapping")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

### Task 7: Homography experiments

1. Move one corner in `dst_quad` and re-run the estimation.
2. Explain why four point correspondences are enough for an exact homography in the noise-free case.
3. Add a fifth point correspondence that is slightly noisy. What changes in the estimated matrix?
4. Compare this result with affine estimation. Which model is more flexible?

## 8. Forward Warping vs. Inverse Warping

When warping images, a transform maps source coordinates to target coordinates:

$$
(x', y') = T(x, y)
$$

### Forward warping

Forward warping sends each source pixel to a target location:

$$
g(x', y') = f(x, y)
$$

Problem: transformed pixels may land between target pixels, leaving holes.

### Inverse warping

Inverse warping visits each target pixel and asks where it came from in the source image:

$$
\begin{bmatrix}
x \\
y \\
1
\end{bmatrix}
\sim
H^{-1}
\begin{bmatrix}
x' \\
y' \\
1
\end{bmatrix}
$$

Then we sample $f(x, y)$ to fill the target pixel.

Advantage: every target pixel gets a chance to receive a value.

Problem: $(x, y)$ may not be integer pixel coordinates, so interpolation is needed.

In [ ]:
def forward_warp_nearest(src_img, H, out_shape):
    # Forward warp an image using nearest-neighbor splatting.
    out_h, out_w = out_shape
    out = np.zeros((out_h, out_w), dtype=float)
    counts = np.zeros((out_h, out_w), dtype=float)

    src_h, src_w = src_img.shape
    yy, xx = np.mgrid[0:src_h, 0:src_w]
    pts = np.column_stack([xx.ravel(), yy.ravel()])
    mapped = apply_homography(H, pts)

    x_round = np.rint(mapped[:, 0]).astype(int)
    y_round = np.rint(mapped[:, 1]).astype(int)
    valid = (x_round >= 0) & (x_round < out_w) & (y_round >= 0) & (y_round < out_h)

    values = src_img.ravel()
    for xr, yr, val in zip(x_round[valid], y_round[valid], values[valid]):
        out[yr, xr] += val
        counts[yr, xr] += 1

    nonzero = counts > 0
    out[nonzero] /= counts[nonzero]
    return out, counts

# Rotate around the image center, then shift so the result is visible.
h, w = image.shape
center_x, center_y = (w - 1) / 2, (h - 1) / 2
H_img = translation(center_x, center_y) @ rotation(np.deg2rad(18)) @ translation(-center_x, -center_y)

forward_img, hit_counts = forward_warp_nearest(image, H_img, image.shape)

plt.imshow(forward_img, cmap="gray", origin="upper", vmin=0, vmax=1)
plt.title("Forward warp with nearest-neighbor splatting")
plt.axis("off")
plt.show()

plt.imshow(hit_counts == 0, cmap="gray", origin="upper")
plt.title("Holes left by forward warping")
plt.axis("off")
plt.show()

### Task 8: Forward warping diagnosis

1. Why do black holes appear after forward warping?
2. Increase the rotation angle. Do the holes become more obvious?
3. What would happen if multiple source pixels map to the same target pixel?

## 9. Bilinear Interpolation

Inverse warping requires sampling the source image at non-integer coordinates.

Let $(x, y)$ be a non-integer coordinate. Define:

$$
x_0 = \lfloor x \rfloor, \quad x_1 = x_0 + 1
$$

$$
y_0 = \lfloor y \rfloor, \quad y_1 = y_0 + 1
$$

Let

$$
\alpha = x - x_0, \quad \beta = y - y_0
$$

The bilinear interpolation value is:

$$
I(x,y) =
(1-\alpha)(1-\beta)I(x_0,y_0)
+ \alpha(1-\beta)I(x_1,y_0)
+ (1-\alpha)\beta I(x_0,y_1)
+ \alpha\beta I(x_1,y_1)
$$

This is linear interpolation in the x direction followed by linear interpolation in the y direction.

In [ ]:
def bilinear_sample(img, x, y, fill_value=0.0):
    # Sample image values at floating-point x and y coordinates using bilinear interpolation.
    h, w = img.shape
    x = np.asarray(x)
    y = np.asarray(y)

    x0 = np.floor(x).astype(int)
    y0 = np.floor(y).astype(int)
    x1 = x0 + 1
    y1 = y0 + 1

    valid = (x0 >= 0) & (x1 < w) & (y0 >= 0) & (y1 < h)

    output = np.full(x.shape, fill_value, dtype=float)
    if not np.any(valid):
        return output

    x0v, x1v = x0[valid], x1[valid]
    y0v, y1v = y0[valid], y1[valid]
    xv, yv = x[valid], y[valid]

    alpha = xv - x0v
    beta = yv - y0v

    Ia = img[y0v, x0v]
    Ib = img[y0v, x1v]
    Ic = img[y1v, x0v]
    Id = img[y1v, x1v]

    output[valid] = ((1 - alpha) * (1 - beta) * Ia +
                     alpha * (1 - beta) * Ib +
                     (1 - alpha) * beta * Ic +
                     alpha * beta * Id)
    return output

# Demonstrate a single bilinear sample.
test_x, test_y = 60.4, 40.7
sample_value = bilinear_sample(image, np.array([test_x]), np.array([test_y]))[0]
print(f"Bilinear sample at x={test_x}, y={test_y}: {sample_value:.4f}")

## 10. Inverse Image Warping

To inverse warp an image with a transform $H$, we iterate over target pixels $(x', y')$, apply $H^{-1}$ to find source coordinates $(x, y)$, and use bilinear interpolation.

$$
\tilde{p}_{source} \sim H^{-1}\tilde{p}_{target}
$$

This is the standard approach used in many image resampling functions.

In [ ]:
def inverse_warp(img, H, out_shape, fill_value=0.0):
    # Inverse warp an image using bilinear interpolation.
    out_h, out_w = out_shape
    H_inv = np.linalg.inv(H)

    yy, xx = np.mgrid[0:out_h, 0:out_w]
    target_pts = np.column_stack([xx.ravel(), yy.ravel()])
    source_pts = apply_homography(H_inv, target_pts)

    samples = bilinear_sample(img, source_pts[:, 0], source_pts[:, 1], fill_value=fill_value)
    return samples.reshape(out_h, out_w)

inverse_img = inverse_warp(image, H_img, image.shape)

plt.imshow(inverse_img, cmap="gray", origin="upper", vmin=0, vmax=1)
plt.title("Inverse warp with bilinear interpolation")
plt.axis("off")
plt.show()

### Task 9: Inverse warping experiments

1. Compare the forward-warp result and inverse-warp result. Which has fewer holes?
2. Change the rotation angle and observe the boundary artifacts.
3. Change `fill_value` to `0.5`. What happens outside the source image support?
4. Modify `bilinear_sample` to implement nearest-neighbor interpolation. Compare sharpness and aliasing.

## 11. Capstone: Synthetic Document Rectification

A common use of homographies is rectifying a tilted planar object, such as a document page.

The goal is to compute a homography from four source corners to four target corners:

$$
\lambda
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}
=
H
\begin{bmatrix}
x \\
y \\
1
\end{bmatrix}
$$

The source corners are the observed quadrilateral. The target corners are the desired rectangle.

This section simulates a document-like rectangle, warps it into a perspective view, and then rectifies it back.

In [ ]:
def make_document_image(height=180, width=260):
    # Create a simple document-like image with border and text-like bars.
    img = np.ones((height, width), dtype=float) * 0.95
    img[10:-10, 10:-10] = 0.85
    img[18:-18, 18:-18] = 0.98

    # Add text-like horizontal bars.
    for y in range(35, 145, 22):
        img[y:y+5, 35:220] = 0.2
        img[y+8:y+11, 35:180] = 0.35

    # Add a small square icon.
    img[120:150, 195:225] = 0.15
    return img

doc = make_document_image()
doc_h, doc_w = doc.shape

rect_corners = np.array([[0.0, 0.0],
                         [doc_w - 1.0, 0.0],
                         [doc_w - 1.0, doc_h - 1.0],
                         [0.0, doc_h - 1.0]])

perspective_corners = np.array([[35.0, 18.0],
                                [225.0, 5.0],
                                [248.0, 165.0],
                                [12.0, 150.0]])

# Build a perspective view by asking each output pixel where it came from in the original document.
H_doc_to_view = estimate_homography(rect_corners, perspective_corners)
view = inverse_warp(doc, np.linalg.inv(H_doc_to_view), doc.shape, fill_value=0.0)

# Estimate the rectifying transform from observed corners back to rectangle corners.
H_view_to_doc = estimate_homography(perspective_corners, rect_corners)
rectified = inverse_warp(view, H_view_to_doc, doc.shape, fill_value=0.0)

plt.imshow(doc, cmap="gray", origin="upper", vmin=0, vmax=1)
plt.title("Original synthetic document")
plt.axis("off")
plt.show()

plt.imshow(view, cmap="gray", origin="upper", vmin=0, vmax=1)
plt.title("Perspective view")
plt.axis("off")
plt.show()

plt.imshow(rectified, cmap="gray", origin="upper", vmin=0, vmax=1)
plt.title("Rectified document")
plt.axis("off")
plt.show()

### Capstone tasks

Complete these tasks after running the document rectification pipeline.

1. **Corner order task:** Swap two points in `perspective_corners`. What breaks? Why does corner order matter?
2. **Model choice task:** Replace the homography with an affine transform estimated from three corners. Which geometric effects cannot be corrected?
3. **Robustness task:** Add random noise to the four observed corners. How sensitive is the rectification?
4. **Real-world design task:** Describe how you would let a user click four corners in an image and then rectify the document.
5. **Reflection task:** Explain why inverse warping is used during rectification instead of forward warping.

## 12. Concept Check

Answer these questions without looking at the code first.

1. What is the difference between changing image intensity values and changing image coordinates?
2. Why do homogeneous coordinates make translation easy to represent?
3. Why does transformation composition order matter?
4. Which transformation family preserves parallel lines?
5. Why does a homography have 8 degrees of freedom instead of 9?
6. Why are at least 3 point correspondences needed for an affine transform?
7. Why are at least 4 point correspondences needed for a homography?
8. What causes holes in forward warping?
9. Why is interpolation necessary in inverse warping?
10. What is the main advantage of bilinear interpolation over nearest-neighbor interpolation?

## 13. Suggested Mini-Projects

Choose one mini-project for deeper practice.

### Mini-project A: Transformation playground

Build a function that accepts sliders or parameters for translation, rotation, scale, and shear. Display the transformed grid and image side by side.

### Mini-project B: Affine alignment

Create two sets of matched points with noise. Estimate an affine transform and visualize prediction errors as arrows.

### Mini-project C: Homography rectification

Use four clicked points from a real image and rectify a planar object. Compare nearest-neighbor and bilinear interpolation.

### Mini-project D: Forward vs. inverse warping report

Create examples where forward warping has holes and inverse warping does not. Explain the result with coordinate diagrams.

## 14. Summary

The core idea of 2D transformations is to describe geometric changes with matrices.

- Linear transformations include scaling, shearing, and rotation.
- Homogeneous coordinates allow translation to be represented as matrix multiplication.
- Affine transformations preserve lines and parallelism.
- Projective transformations preserve lines but not necessarily parallelism.
- Unknown affine transformations can be estimated with linear least squares.
- Image warping requires careful resampling.
- Inverse warping plus interpolation is usually preferred for dense image warping.

Before moving on, make sure you can write the matrix form of translation, rotation, affine transformation, and homography from memory.